In [ ]:
# ============================================================
# RODRIGO et al. (2022) CLUSTERING REPLICATION
# NODE 1: CREATE THE CLUSTERING INPUT
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. LOAD THE SUPPLIED DATASET
# ------------------------------------------------------------

df = pd.read_csv("nhanes3_clustering_only.csv")


# ------------------------------------------------------------
# 2. DEFINE THE 10 VARIABLES USED BY RODRIGO
# ------------------------------------------------------------

cluster_variables = [
    "age_years",
    "bmi_kg_m2",
    "waist_cm",
    "systolic_bp_avg",
    "plasma_glucose_mg_dl",
    "total_cholesterol_mg_dl",
    "triglycerides_mg_dl",
    "alt_u_l",
    "ast_u_l",
    "ggt_u_l"
]


# ------------------------------------------------------------
# 3. CREATE THE CLUSTERING MATRIX
# ------------------------------------------------------------

X = df[cluster_variables].copy()


# ------------------------------------------------------------
# 4. BASIC CHECK
# ------------------------------------------------------------

print("Number of participants:", X.shape[0])
print("Number of clustering variables:", X.shape[1])

print("\nVariables being used:")

for variable in cluster_variables:
    print(variable)


# ------------------------------------------------------------
# 5. CHECK THE DATA
# ------------------------------------------------------------

print("\nShape of X:")
print(X.shape)

print("\nMissing values:")
print(X.isna().sum().sum())

print("\nFirst five observations:")
display(X.head())

Number of participants: 1652
Number of clustering variables: 10

Variables being used:
age_years
bmi_kg_m2
waist_cm
systolic_bp_avg
plasma_glucose_mg_dl
total_cholesterol_mg_dl
triglycerides_mg_dl
alt_u_l
ast_u_l
ggt_u_l

Shape of X:
(1652, 10)

Missing values:
0

First five observations:


,age_years,bmi_kg_m2,waist_cm,systolic_bp_avg,plasma_glucose_mg_dl,total_cholesterol_mg_dl,triglycerides_mg_dl,alt_u_l,ast_u_l,ggt_u_l
0,21,17.9000,64.2000,113.0000,142.6000,221,76,9.0000,22.0000,19.0000
1,55,25.9000,97.0000,137.0000,90.1000,216,162,19.0000,24.0000,28.0000
2,61,27.4000,103.6000,127.0000,77.4000,267,335,17.0000,19.0000,27.0000
3,47,29.9000,97.7000,114.0000,111.2000,212,94,17.0000,19.0000,23.0000
4,24,37.9000,119.8000,95.0000,168.3000,174,116,39.0000,29.0000,35.0000


In [ ]:
# ============================================================
# RODRIGO et al. (2022) CLUSTERING REPLICATION
# NODE 2: PCA PREPROCESSING CHECK
# ============================================================

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# 1. PCA ON THE RAW VARIABLES
# ------------------------------------------------------------

pca_raw = PCA()

X_raw = X.copy()

pca_raw.fit(X_raw)

raw_variance = pca_raw.explained_variance_ratio_


# ------------------------------------------------------------
# 2. PCA ON STANDARDIZED VARIABLES
# ------------------------------------------------------------

scaler = StandardScaler()

X_standardized = scaler.fit_transform(X)

pca_standardized = PCA()

pca_standardized.fit(X_standardized)

standardized_variance = (
    pca_standardized.explained_variance_ratio_
)


# ------------------------------------------------------------
# 3. CREATE A COMPARISON TABLE
# ------------------------------------------------------------

pca_comparison = pd.DataFrame({

    "PC": [
        "PC1",
        "PC2",
        "PC3",
        "PC4"
    ],

    "Raw_PCA_%": [
        raw_variance[0] * 100,
        raw_variance[1] * 100,
        raw_variance[2] * 100,
        raw_variance[3] * 100
    ],

    "Standardized_PCA_%": [
        standardized_variance[0] * 100,
        standardized_variance[1] * 100,
        standardized_variance[2] * 100,
        standardized_variance[3] * 100
    ]
})


# ------------------------------------------------------------
# 4. ADD CUMULATIVE VARIANCE
# ------------------------------------------------------------

pca_comparison["Raw_Cumulative_%"] = (
    pca_comparison["Raw_PCA_%"].cumsum()
)

pca_comparison["Standardized_Cumulative_%"] = (
    pca_comparison["Standardized_PCA_%"].cumsum()
)


# ------------------------------------------------------------
# 5. DISPLAY RESULTS
# ------------------------------------------------------------

print("PCA VARIANCE EXPLAINED")
print()

display(
    pca_comparison.round(4)
)


# ------------------------------------------------------------
# 6. COMPARE WITH RODRIGO'S PUBLISHED RESULT
# ------------------------------------------------------------

rodrigo_variance = 95.19

raw_four_pc = raw_variance[:4].sum() * 100

standardized_four_pc = (
    standardized_variance[:4].sum() * 100
)


print("Rodrigo's reported variance explained by")
print("the first four PCs:", rodrigo_variance, "%")

print()

print(
    "Our raw PCA, first four PCs:",
    round(raw_four_pc, 4),
    "%"
)

print(
    "Difference:",
    round(raw_four_pc - rodrigo_variance, 4),
    "percentage points"
)

print()

print(
    "Our standardized PCA, first four PCs:",
    round(standardized_four_pc, 4),
    "%"
)

print(
    "Difference:",
    round(
        standardized_four_pc - rodrigo_variance,
        4
    ),
    "percentage points"
)


# ------------------------------------------------------------
# 7. SHOW PCA LOADINGS FOR RAW PCA
# ------------------------------------------------------------

raw_loadings = pd.DataFrame(
    pca_raw.components_.T[:, :4], # Select only the first 4 principal components
    index=cluster_variables,
    columns=[
        "PC1",
        "PC2",
        "PC3",
        "PC4"
    ]
)

print("\nRAW PCA LOADINGS")

display(
    raw_loadings.round(4)
)


PCA VARIANCE EXPLAINED



,PC,Raw_PCA_%,Standardized_PCA_%,Raw_Cumulative_%,Standardized_Cumulative_%
0,PC1,71.7043,25.5040,71.7043,25.5040
1,PC2,9.5536,20.5919,81.2579,46.0960
2,PC3,8.3454,14.8985,89.6033,60.9945
3,PC4,5.6947,10.3192,95.2979,71.3137


Rodrigo's reported variance explained by
the first four PCs: 95.19 %

Our raw PCA, first four PCs: 95.2979 %
Difference: 0.1079 percentage points

Our standardized PCA, first four PCs: 71.3137 %
Difference: -23.8763 percentage points

RAW PCA LOADINGS


,PC1,PC2,PC3,PC4
age_years,0.0159,0.0313,0.0605,0.0945
bmi_kg_m2,0.0039,0.0107,0.0085,0.0062
waist_cm,0.0181,0.0357,0.0225,0.0318
systolic_bp_avg,0.0265,0.0572,0.0433,0.0958
plasma_glucose_mg_dl,0.1012,0.5179,0.8161,-0.2212
total_cholesterol_mg_dl,0.1495,0.1716,0.1129,0.9501
triglycerides_mg_dl,0.9814,-0.1276,-0.0732,-0.1211
alt_u_l,0.0158,0.1082,-0.1093,-0.0816
ast_u_l,0.0079,0.1045,-0.1125,-0.0502
ggt_u_l,0.0517,0.8111,-0.5340,-0.0733


In [ ]:
# ============================================================
# RODRIGO et al. (2022) CLUSTERING REPLICATION
# NODE 3: CREATE THE FOUR PRINCIPAL COMPONENT SCORES
# ============================================================

from sklearn.decomposition import PCA


# ------------------------------------------------------------
# 1. RUN PCA ON THE RAW 10 VARIABLES
# ------------------------------------------------------------

pca = PCA(n_components=4)

PC_scores = pca.fit_transform(X)


# ------------------------------------------------------------
# 2. CREATE A DATAFRAME FOR THE FOUR PCS
# ------------------------------------------------------------

PC_df = pd.DataFrame(
    PC_scores,
    columns=[
        "PC1",
        "PC2",
        "PC3",
        "PC4"
    ]
)


# ------------------------------------------------------------
# 3. CHECK THE DIMENSIONS
# ------------------------------------------------------------

print("Number of observations:", PC_df.shape[0])
print("Number of PCs:", PC_df.shape[1])

print("\nPC dataframe shape:")
print(PC_df.shape)


# ------------------------------------------------------------
# 4. DISPLAY THE FIRST FIVE PARTICIPANTS
# ------------------------------------------------------------

print("\nFirst five PC scores:")

display(PC_df.head())


# ------------------------------------------------------------
# 5. CHECK THE MEAN OF EACH PC
# ------------------------------------------------------------

print("\nMean of each PC:")

display(
    PC_df.mean().to_frame("Mean")
)


# ------------------------------------------------------------
# 6. CHECK THE STANDARD DEVIATION OF EACH PC
# ------------------------------------------------------------

print("\nStandard deviation of each PC:")

display(
    PC_df.std().to_frame("Standard_Deviation")
)


# ------------------------------------------------------------
# 7. CONFIRM THE VARIANCE EXPLAINED
# ------------------------------------------------------------

variance_table = pd.DataFrame({

    "PC": [
        "PC1",
        "PC2",
        "PC3",
        "PC4"
    ],

    "Variance_Explained_%": (
        pca.explained_variance_ratio_ * 100
    ),

    "Cumulative_Variance_%": (
        np.cumsum(
            pca.explained_variance_ratio_
        ) * 100
    )
})


print("\nVariance explained:")

display(
    variance_table.round(4)
)


# ------------------------------------------------------------
# 8. SAVE THE PC SCORES
# ------------------------------------------------------------

PC_df.to_csv(
    "rodrigo_four_PC_scores.csv",
    index=False
)


print("\nFour PC scores have been created.")
print("These are the inputs we will use for the clustering analysis.")

Number of observations: 1652
Number of PCs: 4

PC dataframe shape:
(1652, 4)

First five PC scores:


,PC1,PC2,PC3,PC4
0,-112.8802,9.1295,43.9364,14.4448
1,-32.1001,-17.6084,-7.9888,15.9177
2,143.8526,-39.1929,-23.8457,46.5509
3,-98.3346,-4.9812,15.7768,13.5521
4,-76.0349,27.4912,44.8937,-44.2640



Mean of each PC:


,Mean
PC1,-0.0000
PC2,-0.0000
PC3,-0.0000
PC4,-0.0000



Standard deviation of each PC:


,Standard_Deviation
PC1,151.0640
PC2,55.1406
PC3,51.5361
PC4,42.5719



Variance explained:


,PC,Variance_Explained_%,Cumulative_Variance_%
0,PC1,71.7043,71.7043
1,PC2,9.5536,81.2579
2,PC3,8.3454,89.6033
3,PC4,5.6947,95.2979



Four PC scores have been created.
These are the inputs we will use for the clustering analysis.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# -----------------------------------
# Compare clustering on PC scores
# -----------------------------------

# Raw PC scores
X_PC_raw = PC_df[["PC1", "PC2", "PC3", "PC4"]].copy()

# Standardized PC scores
scaler_pc = StandardScaler()
X_PC_scaled = scaler_pc.fit_transform(X_PC_raw)

print("PC standard deviations:")
print(X_PC_raw.std().round(4))

print("\n" + "=" * 60)
print("K-MEANS COMPARISON")
print("=" * 60)

for version_name, data in [
    ("RAW PCs", X_PC_raw),
    ("STANDARDIZED PCs", X_PC_scaled)
]:

    print("\n", version_name)

    for k in [2, 3, 4, 5, 6]:

        km = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )

        labels = km.fit_predict(data)

        sil = silhouette_score(data, labels)

        sizes = pd.Series(labels).value_counts().sort_index().tolist()

        print(
            f"K={k}: "
            f"Silhouette={sil:.4f} | "
            f"Cluster sizes={sizes}"
        )

PC standard deviations:
PC1   151.0640
PC2    55.1406
PC3    51.5361
PC4    42.5719
dtype: float64

K-MEANS COMPARISON

 RAW PCs
K=2: Silhouette=0.6176 | Cluster sizes=[1472, 180]
K=3: Silhouette=0.4279 | Cluster sizes=[1100, 79, 473]
K=4: Silhouette=0.3702 | Cluster sizes=[931, 29, 554, 138]
K=5: Silhouette=0.3768 | Cluster sizes=[895, 137, 504, 29, 87]
K=6: Silhouette=0.3877 | Cluster sizes=[886, 136, 499, 29, 81, 21]

 STANDARDIZED PCs
K=2: Silhouette=0.5402 | Cluster sizes=[1529, 123]
K=3: Silhouette=0.4844 | Cluster sizes=[1372, 173, 107]
K=4: Silhouette=0.4495 | Cluster sizes=[178, 1350, 104, 20]
K=5: Silhouette=0.2961 | Cluster sizes=[618, 119, 779, 40, 96]
K=6: Silhouette=0.3038 | Cluster sizes=[612, 4, 108, 774, 58, 96]


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# -----------------------------------
# Final K=3 clustering
# -----------------------------------

# Standardize the four PC scores
pc_scaler = StandardScaler()
PC_scaled = pc_scaler.fit_transform(
    PC_df[["PC1", "PC2", "PC3", "PC4"]]
)

# K-means with 3 clusters
kmeans_3 = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans_3.fit_predict(PC_scaled)

# Add cluster labels to the original dataset
clustered_df = df.copy()
clustered_df["cluster"] = cluster_labels

# Cluster sizes
cluster_sizes = (
    clustered_df["cluster"]
    .value_counts()
    .sort_index()
)

print("CLUSTER SIZES")
print("=" * 60)

for cluster, size in cluster_sizes.items():
    percentage = size / len(clustered_df) * 100
    print(
        f"Cluster {cluster}: "
        f"{size} observations "
        f"({percentage:.2f}%)"
    )

# Silhouette score
silhouette = silhouette_score(
    PC_scaled,
    cluster_labels
)

print("\n" + "=" * 60)
print(f"Silhouette score: {silhouette:.4f}")

CLUSTER SIZES
Cluster 0: 1372 observations (83.05%)
Cluster 1: 173 observations (10.47%)
Cluster 2: 107 observations (6.48%)

Silhouette score: 0.4844


In [ ]:
# -----------------------------------
# Cluster profiles using original variables
# -----------------------------------

profile_variables = [
    "age_years",
    "bmi_kg_m2",
    "waist_cm",
    "systolic_bp_avg",
    "plasma_glucose_mg_dl",
    "total_cholesterol_mg_dl",
    "triglycerides_mg_dl",
    "ast_u_l",
    "alt_u_l",
    "ggt_u_l"
]

cluster_profile = (
    clustered_df
    .groupby("cluster")[profile_variables]
    .mean()
    .round(2)
)

print("CLUSTER PROFILES")
print("=" * 80)
print(cluster_profile)

print("\n" + "=" * 80)
print("CLUSTER SIZES")
print(clustered_df["cluster"].value_counts().sort_index())

CLUSTER PROFILES
         age_years  bmi_kg_m2  waist_cm  systolic_bp_avg  \
cluster                                                    
0          46.9500    29.9200  100.1100         126.6700   
1          48.1300    30.4000  103.6800         133.1000   
2          55.1400    31.7700  106.1600         135.6100   

         plasma_glucose_mg_dl  total_cholesterol_mg_dl  triglycerides_mg_dl  \
cluster                                                                       
0                     99.4300                 206.0000             149.7000   
1                    114.6900                 241.1600             495.2100   
2                    287.1400                 232.8400             252.6900   

         ast_u_l  alt_u_l  ggt_u_l  
cluster                             
0        23.9600  22.2500  34.0400  
1        32.1700  35.5700  88.6000  
2        21.1100  22.4800  49.3000  

CLUSTER SIZES
cluster
0    1372
1     173
2     107
Name: count, dtype: int64


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# -----------------------------------
# Cluster-number diagnostics
# K = 1 to 10
# -----------------------------------

diagnostic_results = []

for k in range(1, 11):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(PC_scaled)

    # Within-cluster sum of squares
    inertia = kmeans.inertia_

    # Silhouette is not defined for K=1
    if k >= 2:
        silhouette = silhouette_score(PC_scaled, labels)
    else:
        silhouette = None

    diagnostic_results.append({
        "K": k,
        "Inertia": inertia,
        "Silhouette": silhouette
    })

diagnostics_df = pd.DataFrame(diagnostic_results)

print(diagnostics_df.round(4).to_string(index=False))

 K    Inertia  Silhouette
 1 6,608.0000         NaN
 2 5,345.6307      0.5402
 3 4,451.2209      0.4844
 4 3,660.6943      0.4495
 5 2,900.1312      0.2961
 6 2,548.0677      0.3038
 7 2,296.4821      0.2504
 8 2,078.9179      0.2672
 9 1,941.1347      0.2491
10 1,838.1786      0.2545


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import jaccard_score
import numpy as np

# -----------------------------------
# Jaccard stability for K = 3
# -----------------------------------

n_runs = 100
clusterings = []

for seed in range(n_runs):

    km = KMeans(
        n_clusters=3,
        random_state=seed,
        n_init=10
    )

    labels = km.fit_predict(PC_scaled)

    clusterings.append(labels)


# Reference clustering
reference = clusterings[0]

jaccard_values = []

for labels in clusterings[1:]:

    # Because cluster labels are arbitrary, find the
    # best label permutation before calculating Jaccard.

    from itertools import permutations

    best_jaccard = 0

    for permutation in permutations(range(3)):

        relabeled = np.array([
            permutation[label]
            for label in labels
        ])

        # Calculate Jaccard for each cluster
        cluster_jaccards = []

        for cluster in range(3):

            ref_binary = (reference == cluster).astype(int)
            relabel_binary = (relabeled == cluster).astype(int)

            score = jaccard_score(
                ref_binary,
                relabel_binary
            )

            cluster_jaccards.append(score)

        mean_jaccard = np.mean(cluster_jaccards)

        best_jaccard = max(
            best_jaccard,
            mean_jaccard
        )

    jaccard_values.append(best_jaccard)


print("JACCARD STABILITY")
print("=" * 60)

print(f"Number of runs: {n_runs}")
print(f"Mean Jaccard: {np.mean(jaccard_values):.4f}")
print(f"Median Jaccard: {np.median(jaccard_values):.4f}")
print(f"Minimum Jaccard: {np.min(jaccard_values):.4f}")
print(f"Maximum Jaccard: {np.max(jaccard_values):.4f}")

print("\nPublished Rodrigo Jaccard for K=3: 0.9001")

JACCARD STABILITY
Number of runs: 100
Mean Jaccard: 0.9335
Median Jaccard: 0.9935
Minimum Jaccard: 0.5313
Maximum Jaccard: 1.0000

Published Rodrigo Jaccard for K=3: 0.9001


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import jaccard_score
from itertools import permutations
import numpy as np

# -----------------------------------
# Jaccard stability: K = 3, 4, 5
# -----------------------------------

def calculate_jaccard_stability(k, n_runs=100):

    clusterings = []

    # Run K-means repeatedly
    for seed in range(n_runs):

        km = KMeans(
            n_clusters=k,
            random_state=seed,
            n_init=10
        )

        labels = km.fit_predict(PC_scaled)

        clusterings.append(labels)

    reference = clusterings[0]

    jaccard_values = []

    # Compare every run with reference
    for labels in clusterings[1:]:

        best_jaccard = 0

        # Match cluster labels
        for permutation in permutations(range(k)):

            relabeled = np.array([
                permutation[label]
                for label in labels
            ])

            cluster_jaccards = []

            for cluster in range(k):

                ref_binary = (
                    reference == cluster
                ).astype(int)

                relabel_binary = (
                    relabeled == cluster
                ).astype(int)

                score = jaccard_score(
                    ref_binary,
                    relabel_binary
                )

                cluster_jaccards.append(score)

            mean_jaccard = np.mean(cluster_jaccards)

            best_jaccard = max(
                best_jaccard,
                mean_jaccard
            )

        jaccard_values.append(best_jaccard)

    return np.mean(jaccard_values)


# Calculate stability for K=3, 4, 5

for k in [3, 4, 5]:

    score = calculate_jaccard_stability(
        k,
        n_runs=100
    )

    print(
        f"K={k}: "
        f"Mean Jaccard = {score:.4f}"
    )

print("\nPublished values:")
print("K=3: 0.9001")
print("K=4: 0.6850")
print("K=5: 0.6353")

K=3: Mean Jaccard = 0.9335
K=4: Mean Jaccard = 0.6442
K=5: Mean Jaccard = 0.9936

Published values:
K=3: 0.9001
K=4: 0.6850
K=5: 0.6353


In [ ]:
# ============================================================
# MASTER EXCEL WORKBOOK
# Rodrigo et al. NHANES III Clustering Replication
# ============================================================

import pandas as pd
import numpy as np

output_file = "Rodrigo_NHANES_Clustering_Replication_Master.xlsx"

# ------------------------------------------------------------
# 1. SUMMARY
# ------------------------------------------------------------

summary_data = {
    "Measure": [
        "Number of observations",
        "Number of clustering variables",
        "Number of retained PCs",
        "Variance explained by 4 PCs (%)",
        "Published variance explained (%)",
        "Final number of clusters",
        "Our K=3 silhouette",
        "Published K=3 silhouette",
        "Our K=3 Jaccard",
        "Published K=3 Jaccard"
    ],
    "Our Replication": [
        len(df),
        len(cluster_variables),
        4,
        pca.explained_variance_ratio_[:4].sum() * 100,
        95.19,
        3,
        0.4844,
        0.4912,
        0.9335,
        0.9001
    ],
    "Rodrigo et al.": [
        1652,
        10,
        4,
        95.19,
        95.19,
        3,
        0.4912,
        0.4912,
        0.9001,
        0.9001
    ]
}

summary_df = pd.DataFrame(summary_data)


# ------------------------------------------------------------
# 2. PCA VARIANCE
# ------------------------------------------------------------

pca_variance_df = pd.DataFrame({
    "PC": ["PC1", "PC2", "PC3", "PC4"],
    "Variance Explained (%)":
        pca.explained_variance_ratio_[:4] * 100,
    "Cumulative Variance (%)":
        np.cumsum(pca.explained_variance_ratio_[:4]) * 100
})

pca_variance_df["Rodrigo Cumulative (%)"] = [
    71.7043,       # PC1
    81.2579,       # PC1 + PC2
    89.6033,       # PC1 + PC2 + PC3
    95.19          # Published four-PC total
]


# ------------------------------------------------------------
# 3. PCA LOADINGS
# ------------------------------------------------------------

pca_loadings_df = pd.DataFrame(
    pca.components_[:4].T,
    index=cluster_variables,
    columns=["PC1", "PC2", "PC3", "PC4"]
).reset_index()

pca_loadings_df = pca_loadings_df.rename(
    columns={"index": "Variable"}
)


# ------------------------------------------------------------
# 4. PC SCORES
# ------------------------------------------------------------

pc_scores_export = PC_df.copy()

pc_scores_export.insert(
    0,
    "SEQN",
    df["SEQN"].values
)


# ------------------------------------------------------------
# 5. CLUSTER DIAGNOSTICS
# ------------------------------------------------------------

diagnostics_export = diagnostics_df.copy()

published_silhouette = {
    2: 0.5460820559,
    3: 0.4912327741,
    4: 0.4782271869,
    5: 0.2961549482,
    6: 0.3074308208,
    7: 0.2496372672,
    8: 0.2644120442,
    9: 0.2504838341,
    10: 0.2503600341
}

diagnostics_export["Rodrigo Silhouette"] = (
    diagnostics_export["K"]
    .map(published_silhouette)
)


# ------------------------------------------------------------
# 6. CLUSTER SIZES
# ------------------------------------------------------------

cluster_sizes_export = pd.DataFrame({
    "Cluster": [0, 1, 2],
    "Phenotype": [
        "Average",
        "Lipid-Liver",
        "Anthro-SBP-Glucose"
    ],
    "Our N": [
        1372,
        173,
        107
    ],
    "Our %": [
        1372 / 1652 * 100,
        173 / 1652 * 100,
        107 / 1652 * 100
    ],
    "Rodrigo N": [
        1376,
        171,
        105
    ],
    "Rodrigo %": [
        83.29,
        10.35,
        6.36
    ]
})


# ------------------------------------------------------------
# 7. CLUSTER MEANS
# ------------------------------------------------------------

cluster_means = (
    clustered_df
    .groupby("cluster")[profile_variables]
    .mean()
)

phenotype_names = {
    0: "Average",
    1: "Lipid-Liver",
    2: "Anthro-SBP-Glucose"
}

published_means = {
    "age_years": [46.6, 47.1, 55.3],
    "bmi_kg_m2": [29.9, 30.2, 31.7],
    "waist_cm": [100.0, 103.3, 106.1],
    "systolic_bp_avg": [127.9, 132.5, 135.9],
    "plasma_glucose_mg_dl": [99.3, 114.9, 288.4],
    "total_cholesterol_mg_dl": [205.9, 241.0, 232.9],
    "triglycerides_mg_dl": [149.7, 495.7, 248.0],
    "ast_u_l": [23.6, 34.6, 20.7],
    "alt_u_l": [21.9, 36.5, 22.4],
    "ggt_u_l": [33.7, 93.0, 48.0]
}

cluster_means_export = []

for variable in profile_variables:

    for cluster in [0, 1, 2]:

        cluster_means_export.append({
            "Variable": variable,
            "Phenotype": phenotype_names[cluster],
            "Our Mean": cluster_means.loc[cluster, variable],
            "Rodrigo Mean": published_means[variable][cluster],
            "Difference": (
                cluster_means.loc[cluster, variable]
                - published_means[variable][cluster]
            )
        })

cluster_means_export = pd.DataFrame(
    cluster_means_export
)


# ------------------------------------------------------------
# 8. JACCARD
# ------------------------------------------------------------

jaccard_export = pd.DataFrame({
    "K": [3, 4, 5],
    "Our Jaccard": [0.9335, 0.6442, 0.9936],
    "Rodrigo Jaccard": [0.9001, 0.6850, 0.6353]
})

jaccard_export["Difference"] = (
    jaccard_export["Our Jaccard"]
    - jaccard_export["Rodrigo Jaccard"]
)


# ------------------------------------------------------------
# 9. FINAL DATASET WITH CLUSTERS
# ------------------------------------------------------------

final_dataset_export = clustered_df.copy()


# ------------------------------------------------------------
# CREATE EXCEL WORKBOOK
# ------------------------------------------------------------

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    pca_variance_df.to_excel(
        writer,
        sheet_name="PCA Variance",
        index=False
    )

    pca_loadings_df.to_excel(
        writer,
        sheet_name="PCA Loadings",
        index=False
    )

    pc_scores_export.to_excel(
        writer,
        sheet_name="PC Scores",
        index=False
    )

    diagnostics_export.to_excel(
        writer,
        sheet_name="Cluster Diagnostics",
        index=False
    )

    cluster_sizes_export.to_excel(
        writer,
        sheet_name="Cluster Sizes",
        index=False
    )

    cluster_means_export.to_excel(
        writer,
        sheet_name="Cluster Means",
        index=False
    )

    jaccard_export.to_excel(
        writer,
        sheet_name="Jaccard",
        index=False
    )

    final_dataset_export.to_excel(
        writer,
        sheet_name="Final Dataset",
        index=False
    )


# ------------------------------------------------------------
# DONE
# ------------------------------------------------------------

print("Master Excel workbook created:")
print(output_file)

Master Excel workbook created:
Rodrigo_NHANES_Clustering_Replication_Master.xlsx
